In [1]:
import pandas as pd

In [2]:
%pip install transformers datasets scikit-learn torch accelerate

In [3]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ──────────────────────────────────────────────
# 1. CONFIG
# ──────────────────────────────────────────────
MODEL_NAME   = "prajjwal1/bert-mini"
NUM_LABELS   = 8          # ← change to your number of classes
MAX_LEN      = 512        # max token length
BATCH_SIZE   = 32
EPOCHS       = 50
LR           = 5e-5
WARMUP_RATIO = 0.1        # fraction of steps used for LR warm-up
WEIGHT_DECAY = 0.01
SEED         = 42
SAVE_DIR     = "./bert_mini_classifier"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

torch.manual_seed(SEED)
np.random.seed(SEED)

local_dataset_path = "data/conexao-labels-full-4.1nano-p2.csv"
online_dataset_path = "https://media.githubusercontent.com/media/kamilyassis/pln/refs/heads/main/2026-03-29_label_datasets/data/out/conexao-labels-full-4.1nano-p2.csv"
in_colab = True

DATASET_ARGS = {
    "filepath_or_buffer": local_dataset_path if not in_colab else online_dataset_path,
    "sep": '|'
}

Using device: cuda


In [4]:

# ──────────────────────────────────────────────
# 2. DATASET
# ──────────────────────────────────────────────
class TextClassificationDataset(Dataset):
    """
    Generic dataset for text classification.

    Args:
        texts  : list[str]  – raw input sentences
        labels : list[int]  – integer class indices (0 … NUM_LABELS-1)
        tokenizer           – HuggingFace tokenizer
        max_len : int       – max sequence length
    """

    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


In [5]:


# ──────────────────────────────────────────────
# 3. METRICS
# ──────────────────────────────────────────────
def compute_metrics(preds, labels):
    preds = np.argmax(preds, axis=1)
    acc   = accuracy_score(labels, preds)
    report = classification_report(labels, preds, zero_division=0)
    return acc, report


# ──────────────────────────────────────────────
# 4. TRAIN ONE EPOCH
# ──────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0

    for batch in tqdm(loader, desc="Training batches"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)


# ──────────────────────────────────────────────
# 5. EVALUATE
# ──────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            total_loss += outputs.loss.item()
            all_preds.append(outputs.logits.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    acc, report = compute_metrics(all_preds, all_labels)
    return total_loss / len(loader), acc, report


# ──────────────────────────────────────────────
# 6. INFERENCE / PREDICT
# ──────────────────────────────────────────────
def predict(texts, model, tokenizer, id2label=None):
    """
    Run inference on a list of strings.

    Returns:
        predicted class indices (and labels if id2label is provided)
    """
    model.eval()
    all_preds = []

    dataset = TextClassificationDataset(
        texts, [0] * len(texts), tokenizer  # dummy labels
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            outputs  = model(input_ids=input_ids, attention_mask=attention_mask)
            preds    = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)

    if id2label:
        return [id2label[p] for p in all_preds]
    return all_preds



In [6]:

# ── 7a. Load your data ──────────────────────
# Replace this block with your actual data loading logic.
# `texts`  → list of raw strings
# `labels` → list of int class indices (0-based)

df = pd.read_csv(**DATASET_ARGS)
texts  = df['sentence'].tolist()  # or 'sentence' if that's your column
labels = df['label'].tolist() # make sure these are integers from 0 to NUM_LABELS-1

# Optionally define a human-readable mapping (used at inference time)
id2label = {
    0: "No_Label",
    1: "Loaded_Language",
    2: "Name_Calling-Labeling",
    3: "Doubt",
    4: "Repetition",
    5: "Appeal_to_Fear-Prejudice",
    6: "Flag_Waving",
    7: "Exaggeration-Minimisation"
}
label2id = {v: k for k, v in id2label.items()}

labels = [label2id[l] for l in labels]  # convert to integer indices if needed

from imblearn.over_sampling import RandomOverSampler

# Create 2D array for texts (required by imbalanced-learn)
texts_array = np.array(texts).reshape(-1, 1)

# Oversample to balance all classes
ros = RandomOverSampler(random_state=SEED)
texts_resampled, labels_resampled = ros.fit_resample(texts_array, labels)

# Garantir conversão segura
texts = [str(t).strip() for t in texts_resampled.squeeze()] if isinstance(texts_resampled, np.ndarray) else texts_resampled
labels = list(labels_resampled) if not isinstance(labels_resampled, list) else labels_resampled

# ── 7b. Train / val / test split ────────────
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)

In [7]:
train_texts[:5]

['Esta elite islâmica do Norte, é a principal responsável pela introdução da sharia nestas doze províncias.',
 "Quantas vezes pessoas já nos disseram que se tivéssemos 'falado direito' tudo teria sido diferente.",
 'Sim, Dr. Merha, não é mais eficaz, pois são doentes graves, já no estágio 3 hiperinflamatório da doença.',
 'Trump chegou a subir tarifas de vários países, só pra depois baixar tudo e equalizar em um mesmo patamar — menos a China.',
 'Mas o grande culpa é o Bolsonaro\nA politização das grandes redações de jornais brasileiros segue a pleno vapor, na tentativa de omitir dados, ocorrências e números — alimentando narrativas que servem como palanque eleitoral para difundir conteúdos a bel prazer.']

In [8]:

# ── 7c. Tokenizer & model ───────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
).to(DEVICE)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-mini
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

In [9]:

# ── 7d. DataLoaders ─────────────────────────
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer)
val_dataset   = TextClassificationDataset(val_texts,   val_labels,   tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)


In [10]:

# ── 7e. Optimizer & scheduler ───────────────
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)


In [11]:

# ── 7f. Training loop ───────────────────────
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_acc, val_report = evaluate(model, val_loader)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )
    print(val_report)

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs(SAVE_DIR, exist_ok=True)
        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f"  ✓ New best model saved to {SAVE_DIR}")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")


Training batches: 100%|██████████| 1012/1012 [09:06<00:00,  1.85it/s]


Epoch 1/50 | Train Loss: 2.0509 | Val Loss: 1.9629 | Val Acc: 0.2705
              precision    recall  f1-score   support

           0       0.19      0.09      0.12      1012
           1       0.22      0.27      0.24      1012
           2       0.26      0.30      0.28      1012
           3       0.45      0.39      0.42      1012
           4       0.51      0.25      0.33      1012
           5       0.28      0.20      0.23      1012
           6       0.25      0.21      0.23      1012
           7       0.20      0.46      0.28      1012

    accuracy                           0.27      8096
   macro avg       0.30      0.27      0.27      8096
weighted avg       0.30      0.27      0.27      8096



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ New best model saved to ./bert_mini_classifier


Training batches:  10%|▉         | 99/1012 [00:55<08:27,  1.80it/s]


KeyboardInterrupt: 

In [ ]:

# ── 7g. Inference example ───────────────────
best_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(DEVICE)
sample     = ["Você prefere o brasil ou cuba?", "Caralho porra que merda de politica do caralho"]
preds      = predict(sample, best_model, tokenizer, id2label=id2label)
for text, pred in zip(sample, preds):
    print(f"  [{pred}] {text}")

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

  [No_Label] This film blew me away!
  [No_Label] Absolutely dreadful.


In [ ]:
# save model to google drive from colab
from google.colab import drive
drive.mount("/content/drive")
model.save_pretrained("/content/drive/MyDrive/bert_mini_classifier")
tokenizer.save_pretrained("/content/drive/MyDrive/bert_mini_classifier")